# 03 — Triple-Barrier Labeling

**Purpose:** Apply triple-barrier labeling to every M1 bar.

For each bar at time `t`, three barriers are placed:
- **Upper barrier (TP):** price_t + `tp_atr × ATR`  
- **Lower barrier (SL):** price_t - `sl_atr × ATR`  
- **Horizontal barrier:** `t + horizon` bars  

The label is determined by which barrier is touched **first** in bars `[t+1, t+horizon]`:
- `+1`: upper touched first → long signal
- `-1`: lower touched first → short signal  
- `0`:  neither touched in time → neutral / no-trade

Additional metadata per label:
- `mfe` (max favourable excursion)
- `mae` (max adverse excursion)
- `time_to_exit` (bars until barrier hit or horizon)
- `barrier_side` (TP / SL / TIMEOUT)

**Inputs:** `data/features/{symbol}_features.parquet`  
**Outputs:** `data/features/{symbol}_labeled_{config_key}.parquet`

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.utils.config import load_config, get_symbol, get_paths, get_labeling_params
from src.labels.triple_barrier import (
    apply_triple_barrier, label_statistics, build_labeled_dataset
)

cfg      = load_config()
paths    = get_paths(cfg, "..")
symbol   = get_symbol(cfg)
lbl_cfg  = get_labeling_params(cfg)

print(f"Labeling config:\n{lbl_cfg}")

## 1. Load features

In [ ]:
feat_path = paths["features"] / f"{symbol}_features.parquet"
features  = pd.read_parquet(feat_path)
print(f"Features loaded: {features.shape}")

# Extract required series
prices = features["close"]
atr    = features["atr_14"]

print(f"Price range: {prices.min():.1f} – {prices.max():.1f}")
print(f"ATR range:   {atr.min():.2f} – {atr.max():.2f} (mean: {atr.mean():.2f})")

## 2. Label with baseline parameters

Start with a single TP/SL/horizon combination to understand the label distribution.

In [ ]:
# Baseline: TP=2.0×ATR, SL=1.0×ATR, horizon=30 min
TP_ATR   = 2.0
SL_ATR   = 1.0
HORIZON  = 30

long_labels  = apply_triple_barrier(prices, atr, TP_ATR, SL_ATR, HORIZON, side= 1)
short_labels = apply_triple_barrier(prices, atr, TP_ATR, SL_ATR, HORIZON, side=-1)

print("Long label statistics:")
for k, v in label_statistics(long_labels).items():
    print(f"  {k}: {v}")

print("\nShort label statistics:")
for k, v in label_statistics(short_labels).items():
    print(f"  {k}: {v}")

In [ ]:
# Plot label distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, lbl_df, title in [
    (axes[0], long_labels,  "Long Labels"),
    (axes[1], short_labels, "Short Labels"),
]:
    counts = lbl_df["label"].value_counts().sort_index()
    colors = {-1: "red", 0: "grey", 1: "green"}
    ax.bar(
        [str(k) for k in counts.index],
        counts.values,
        color=[colors.get(k, "blue") for k in counts.index],
        alpha=0.8
    )
    ax.set_title(f"{title} (TP={TP_ATR}×ATR, SL={SL_ATR}×ATR, H={HORIZON})")
    ax.set_xlabel("Label")
    ax.set_ylabel("Count")
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("../reports/label_distribution.png", dpi=120)
plt.show()

## 3. MFE / MAE analysis

Understanding the maximum favourable and adverse excursions helps set realistic TP/SL levels.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# MFE distribution (winners)
win_mask = long_labels["label"] == 1
axes[0].hist(long_labels.loc[win_mask, "mfe"], bins=50, color="green", alpha=0.7)
axes[0].set_title("MFE Distribution (Long Winners)")
axes[0].set_xlabel("Max Favourable Excursion (points)")

# MAE distribution (all trades)
axes[1].hist(long_labels["mae"].clip(lower=-500), bins=50, color="red", alpha=0.7)
axes[1].set_title("MAE Distribution (All Long Bars)")
axes[1].set_xlabel("Max Adverse Excursion (points)")

plt.tight_layout()
plt.savefig("../reports/mfe_mae_distribution.png", dpi=120)
plt.show()

print("MFE stats (winners):")
print(long_labels.loc[win_mask, "mfe"].describe())
print("\nMAE stats (all bars):")
print(long_labels["mae"].describe())

## 4. Build labeled dataset and save

In [ ]:
# Build combined dataset for baseline parameters
labeled = build_labeled_dataset(features, prices, atr, TP_ATR, SL_ATR, HORIZON)

print(f"Labeled dataset shape: {labeled.shape}")

# Save
key = f"tp{TP_ATR}_sl{SL_ATR}_h{HORIZON}"
out_path = paths["features"] / f"{symbol}_labeled_{key}.parquet"
labeled.to_parquet(out_path, engine="pyarrow", compression="snappy")
print(f"Saved → {out_path}")

## 5. Time-to-exit analysis

How quickly do trades typically resolve? This informs our max holding time setting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, lbl_df, title in [
    (axes[0], long_labels[long_labels["label"] != 0],  "Long (TP+SL)"),
    (axes[1], short_labels[short_labels["label"] != 0], "Short (TP+SL)"),
]:
    ax.hist(lbl_df["time_to_exit"], bins=30, color="steelblue", alpha=0.8)
    ax.set_title(f"Time to Exit — {title}")
    ax.set_xlabel("Bars (M1 minutes)")
    ax.set_ylabel("Count")
    ax.axvline(lbl_df["time_to_exit"].median(), color="red", ls="--", label="Median")
    ax.legend()

plt.tight_layout()
plt.savefig("../reports/time_to_exit.png", dpi=120)
plt.show()